In [ ]:
import jax 
import jax.numpy as jnp
from flax import linen as nn

def get_layers(neurons: tuple[int, ...]):

    if not neurons:
        return None
    
    layers = []
    for n in neurons:
        layers.append(
            nn.Dense(n)
        )
    return tuple(layers)

def forward(h, last_linear, layers):
    if not layers: 
        return h
    
    for layer in layers[:-1]:
        h = nn.relu(layer(h))
    
    final_layer = layers[-1]
    h = final_layer(h)
    
    return h if last_linear else nn.relu(h)   

class DeModelMlp(nn.Module):
    
    out: tuple[int, ...]
    delegation: tuple[int, ...]
    body: tuple[int, ...] | None = None
    
    def setup(self):
        self.body_layers = get_layers(self.body)
        self.out_layers = get_layers(self.out)
        self.delegation_layers = get_layers(self.delegation)

    def __call__(self, x: jax.Array):
        h = x
        h_body = forward(h, last_linear=False, layers=self.body_layers)
        y = forward(h_body, last_linear=True, layers=self.out_layers)
        d = forward(h_body, last_linear=True, layers=self.delegation_layers)
        return y, d

class LeMlp(nn.Module):
    n_models: int
    out: tuple[int, ...]
    delegation: tuple[int, ...]
    body: tuple[int, ...] | None = None

    def setup(self):
        
        assert self.delegation[-1] == self.n_models, "Out of delegation needs to equal to n_models"

        VmappedDeModelMlp = nn.vmap(
            DeModelMlp,
            variable_axes={'params': 0},
            split_rngs={'params': True}, # Vmap over different models
            in_axes=None, # Do not vmap over batch elements
            axis_size=self.n_models,
            out_axes=1 # Stack the model outputs to axis=1
        )
        
        self.ensemble = VmappedDeModelMlp(
            out=self.out, 
            delegation=self.delegation, 
            body=self.body
        )

    def __call__(self, x: jax.Array):
        return self.ensemble(x)

In [24]:

k1, k2 = jax.random.split(jax.random.key(123))

x = jax.random.uniform(k1, (32, 3))

e = LeMlp(
    n_models=9,
    body=(10,),
    out=(10, 5),
    delegation=(9,)  # Added trailing comma
)
params = e.init(k2, x[[0], :])["params"]

out = e.apply({"params": params}, x)  # Passed x to apply


In [32]:
out[1][2]

Array([[ 2.43651062e-01,  3.25399846e-01,  2.45215982e-01,
        -1.51339337e-01,  4.47084531e-02,  3.19936275e-02,
         7.51890689e-02,  1.29513144e-01, -3.07852268e-01],
       [ 3.33973497e-01,  5.50473273e-01,  2.77067870e-01,
        -1.89043671e-01,  3.80693711e-02,  5.58309406e-02,
         1.33681610e-01,  2.45153368e-01, -5.53084016e-01],
       [ 9.58338976e-02,  8.04909527e-01,  8.64893198e-04,
         2.94821233e-01,  1.72077790e-02,  2.05883801e-01,
        -2.50082016e-02,  1.47514701e-01, -7.04494834e-01],
       [ 8.30525994e-01,  1.20956469e+00,  7.27195144e-01,
        -3.76859426e-01,  1.56129956e-01,  3.66317153e-01,
         4.23974037e-01,  2.97265410e-01, -1.13230610e+00],
       [ 1.18144810e-01,  1.08232164e+00, -2.14539468e-02,
         5.73526084e-01,  8.47710967e-02,  5.87218106e-01,
         8.55676234e-02, -1.14739358e-01, -8.58933806e-01],
       [ 6.95456386e-01,  1.05535829e+00,  5.87647915e-01,
        -3.60512733e-01,  1.10603638e-01,  2.384294